In [9]:
import Implementation_OFLP as IO
import run_OFLP as rO
import networkx as nx
a = rO.read_instance('ofl_2_100i.gms')


def get_feas_points(circles_used : list, x_lb, x_ub, y_lb, y_ub):
    # We now look at all of the intersections between the circles and the square
    n = len(circles_used)
    m = len(circles_used) + 4
    points = [[x_lb, y_lb], [x_lb, y_ub], [x_ub, y_lb], [x_ub, y_ub]]
    points_belong_circle = [[]]*m
    for i in range(len(circles_used)):
        point, points_belong_circle = IO.box_circles(x_lb, x_ub, y_lb, y_ub, circles_used, i, points_belong_circle)
        #print(f"We are looking at the circle: {circle} and it produces points :{point}")
        for p in point:
            #print(f"Point {p} is not in the list of points: {points}? {p not in points}")
            if type(p) != list:
                n_point = [p]
                p = n_point
            if p not in points_belong_circle[i+4]:
                points_belong_circle[i+4].append(p)
            if p not in points:
                points.append(p)

    # We now look at all of the intersections between the circles
    for i in range(n):
        for j in range(i + 1,n):
            c1, c2 = circles_used[i], circles_used[j]
            if c1 == c2:
                continue
            elif ((c1[0] - c2[0]) ** 2) + ((c1[1] - c2[1]) ** 2) >= c1[2] ** 2 + c2[2] ** 2:
                continue
            else:
                intersections = IO.circles_circles(circles_used,i,j)
            for point in intersections:
                if type(point) != list:
                    n_point = [point]
                    point = n_point
                if point not in points:
                    interior = False
                    if (x_lb <= point[0] <= x_ub) and (y_lb <= point[1] <= y_ub):
                        for c in circles_used:
                            if IO.interior_circle(point, c) == True:
                                interior = True
                        if (not interior):
                            points_belong_circle[i+4].append(point)
                            points_belong_circle[j+4].append(point)
                            points.append(point)
    return points

def is_feas_solution(points, fac_fac_dist, facilities) -> float:
    G = nx.Graph()
    n = len(points)
    for i in range(n):
        for j in range(i+1, n):
            if rO.distance(points[i], points[j]) >= fac_fac_dist:
                G.add_edge(i, j)
    if nx.max_weight_clique(G, weight=None)[1] >= facilities:
        print(nx.max_weight_clique(G, weight=None))
        return True
    else:
        return False



def create_init_sol(facilities, communities, x_lb, x_ub, y_lb, y_ub, fac_fac_dist, r_guess : float=0) -> float:
    # The idea of the algorithm is to decrease the radius until we find a feasible radius, this way we create a good intial solution.
    feas_sol_found = False
    max_iter = 1e3 * len(communities)
    i = 0
    if r_guess == 0:
        r = 100 / len(communities)
    else:
        r = r_guess
    while (not feas_sol_found) and (i < max_iter):
        circles = []
        for i in communities:
            circles.append([i[0], i[1], r])
        points = get_feas_points(circles, x_lb, x_ub, y_lb, y_ub)

        # We now try to fit the facilities in the points while maintaining the fac_fac_distance
        feas = is_feas_solution(points, fac_fac_dist, facilities)
        if not feas:
            r *= 0.95
            i += 1
        else:
            feas_sol_found = True
            return r
    return r


create_init_sol(a['facilities'], a['communities'], a['x_bounds'][0], a['x_bounds'][1], a['y_bounds'][0], a['y_bounds'][1], a['ff_sq_bounds'][0])

([111, 62, 3, 2], 4)


1.0